# 02 - Template Engines (提示模板引擎)

## 学习目标
- 使用 Jinja2 构建可复用的提示模板
- 掌握 System Prompt 和 User Prompt 的结构化设计
- 实现角色框架模式和动态 Few-shot 注入
- 理解模板继承、组合和常见反模式

In [ ]:
# 初始化环境
import os
import json
import hashlib
from typing import Any
from datetime import datetime
from jinja2 import Environment, BaseLoader, Template, select_autoescape, UndefinedError

# 创建 Jinja2 环境
jinja_env = Environment(
    loader=BaseLoader(),  # 从字符串加载模板
    autoescape=select_autoescape(),
    trim_blocks=True,          # 自动去除块标签后的第一个换行
    lstrip_blocks=True,        # 自动去除块标签前的空白
)

print("Jinja2 环境已初始化")
print(f"Jinja2 版本: {__import__('jinja2').__version__}")

---
## 1. Jinja2 模板基础

### 三大核心语法
| 语法 | 用途 | 示例 |
|------|------|------|
| `{{ variable }}` | 变量输出 | `{{ user_name }}` |
| `{% if/for %}` | 控制流 | `{% if score > 80 %}...{% endif %}` |
| `{# comment #}` | 注释 | `{# 这是注释 #}` |

In [ ]:
# ============================================================
# Jinja2 基础语法演示
# ============================================================

# 1. 变量渲染
template_var = jinja_env.from_string("""
你是一位名叫 {{ name }} 的 {{ role }}，拥有 {{ years }} 年工作经验。
你的专业领域包括：{{ expertise | join('、') }}。
""")

result_var = template_var.render(
    name="张工",
    role="资深后端工程师",
    years=10,
    expertise=["Python", "分布式系统", "数据库优化", "微服务架构"],
)
print("=== 变量渲染 ===")
print(result_var)

# 2. 条件判断
template_if = jinja_env.from_string("""
{% if question_type == 'code' %}
请提供代码示例，并包含注释说明。
输出格式：先解释思路，再给出代码，最后给出复杂度分析。
{% elif question_type == 'concept' %}
请用通俗易懂的语言解释概念，避免使用过于专业的术语。
输出格式：定义 → 类比 → 示例。
{% elif question_type == 'debug' %}
请先分析错误信息，然后定位可能的原因，最后给出修复建议。
输出格式：错误分析 → 根因定位 → 修复方案。
{% else %}
请直接回答问题，保持简洁清晰。
{% endif %}
""")

print("\n=== 条件判断 ===")
for qtype in ["code", "concept", "debug", "unknown"]:
    print(f"\n当 question_type='{qtype}':")
    print(template_if.render(question_type=qtype))

# 3. 循环遍历
template_for = jinja_env.from_string("""
请根据以下{{ items | length }}条用户反馈，总结产品的主要问题：

{% for item in items %}
{{ loop.index }}. [{{ item.sentiment }}] {{ item.text }}
{% endfor %}

请按严重程度排序，并给出改进建议。
""")

feedbacks = [
    {"sentiment": "负面", "text": "APP 频繁闪退，影响使用"},
    {"sentiment": "负面", "text": "支付流程太复杂，要跳转好几次"},
    {"sentiment": "中性", "text": "界面还可以，但加载速度偏慢"},
    {"sentiment": "正面", "text": "客服响应很快，问题解决及时"},
]

print("\n=== 循环遍历 ===")
print(template_for.render(items=feedbacks))

In [ ]:
# ============================================================
# Jinja2 过滤器和自定义函数
# ============================================================

# 自定义过滤器：截断长文本
def truncate_filter(text: str, length: int = 100, suffix: str = "...") -> str:
    """截断长文本，保持完整句子。"""
    if len(text) <= length:
        return text
    return text[:length].rsplit("。", 1)[0] + suffix

jinja_env.filters['truncate'] = truncate_filter

# 自定义过滤器：格式化日期
def date_format(value: str, fmt: str = "%Y年%m月%d日") -> str:
    """格式化日期字符串。"""
    try:
        dt = datetime.fromisoformat(value)
        return dt.strftime(fmt)
    except (ValueError, TypeError):
        return str(value)

jinja_env.filters['date_format'] = date_format

# 测试过滤器和条件渲染
template_filters = jinja_env.from_string("""
合同编号：{{ contract_id }}
签署日期：{{ sign_date | date_format }}
内容摘要：{{ content | truncate(80) }}

{% if contract_type == 'NDA' %}
保密等级：{{ confidentiality | default('standard') | upper }}
{% endif %}

{% set critical_clauses = clauses | selectattr('is_critical') | list %}
{% if critical_clauses %}
关键条款（{{ critical_clauses | length }}条）：
{% for clause in critical_clauses %}
  - 第{{ clause.number }}条：{{ clause.title }}
{% endfor %}
{% else %}
未发现特别关键条款。
{% endif %}
""")

context = {
    "contract_id": "CT-2026-00158",
    "sign_date": "2026-05-15T10:30:00",
    "content": "本合同就双方合作事宜达成以下一致。甲方负责提供技术支持服务，乙方按时支付相应费用。如有一方违约，需按合同第十五条处理。",
    "contract_type": "NDA",
    "confidentiality": "high",
    "clauses": [
        {"number": 3, "title": "保密义务", "is_critical": True},
        {"number": 7, "title": "违约金条款", "is_critical": True},
        {"number": 12, "title": "争议解决", "is_critical": False},
    ],
}

print("=== 过滤器和高级渲染 ===")
print(template_filters.render(**context))

---
## 2. System Prompt 与 User Prompt 的结构化设计

### 分层模型
```
┌─────────────────────────────────────────┐
│  System Prompt (系统提示)                │
│  - 角色定义（Role）                      │
│  - 行为约束（Constraints）               │
│  - 输出规范（Output Spec）               │
│  - 工具定义（Tool Definitions）          │
├─────────────────────────────────────────┤
│  User Prompt (用户提示)                  │
│  - 任务上下文（Context）                 │
│  - 具体输入（Input）                     │
│  - 示例数据（Examples, 若使用 Few-shot） │
│  - 指令补充（Instructions）              │
└─────────────────────────────────────────┘
```

### 职责分离
- **System**: "我是谁" + "我能做什么" + "我怎么做事" — 跨轮次持久
- **User**: "这次要做什么" + "这是输入" — 每轮独立

In [ ]:
# ============================================================
# System/User Prompt 结构化模板
# ============================================================

class PromptTemplate:
    """
    提示模板类：分离 System 和 User 提示构建。
    
    What: 将提示构建逻辑封装为可复用的模板类。
    Why: 分离关注点 — System 定义了模型"是什么"，User 定义"做什么"。
    When: 构建任何 LLM 应用时都应使用此类结构。
    """
    
    def __init__(
        self,
        system_template: str,
        user_template: str,
        system_defaults: dict | None = None,
    ):
        self._system_tpl = jinja_env.from_string(system_template)
        self._user_tpl = jinja_env.from_string(user_template)
        self._system_defaults = system_defaults or {}
    
    def build(self, user_context: dict) -> tuple[str, str]:
        """
        构建 System Prompt 和 User Prompt。
        
        Args:
            user_context: 用户上下文变量
        Returns:
            (system_prompt, user_prompt) 元组
        """
        system_prompt = self._system_tpl.render(**self._system_defaults, **user_context)
        user_prompt = self._user_tpl.render(**user_context)
        return system_prompt, user_prompt
    
    def __repr__(self) -> str:
        return f"PromptTemplate(system={self._system_tpl}, user={self._user_tpl})"

# 法律合同审查模板
legal_system_tpl = """你是一位拥有{{ years }}年经验的{{ role }}。

行为准则：
{% for rule in rules %}
- {{ rule }}
{% endfor %}

审查范围：
{% for scope_item in review_scope %}
- {{ scope_item }}
{% endfor %}

输出规范：
1. 先给出总体风险评估（低/中/高风险）
2. 按条款逐一分析
3. 给出修改建议和替代条款
4. 使用专业但易懂的语言"""

legal_user_tpl = """请审查以下{{ contract_type }}：

合同信息：
- 甲方：{{ party_a }}
- 乙方：{{ party_b }}
- 金额：{{ amount }}
- 期限：{{ duration }}

合同内容：
{{ contract_text }}

{% if special_concerns %}
特别关注：
{% for concern in special_concerns %}
- {{ concern }}
{% endfor %}
{% endif %}"""

legal_template = PromptTemplate(
    system_template=legal_system_tpl,
    user_template=legal_user_tpl,
    system_defaults={
        "role": "公司法务律师",
        "years": 15,
        "rules": [
            "审查必须基于《民法典》和最新司法解释",
            "对可能的法律风险必须明确指出",
            "不确定的问题必须标注为'需进一步核实'",
        ],
        "review_scope": [
            "合同主体资格",
            "权利义务对等性",
            "违约责任条款",
            "争议解决条款",
        ],
    },
)

sys, user = legal_template.build({
    "contract_type": "软件开发合同",
    "party_a": "某科技有限公司",
    "party_b": "某软件开发公司",
    "amount": "500,000 元人民币",
    "duration": "6 个月",
    "contract_text": "第一条 项目范围... 第二条 付款方式...（合同文本略）",
    "special_concerns": ["知识产权归属", "源代码交付标准"],
})

print("=== System Prompt ===")
print(sys)
print("\n=== User Prompt ===")
print(user)

---
## 3. 角色框架模式 (Role Framing Patterns)

角色框架通过 System Prompt 设定模型的行为预期，是控制输出质量和风格的最有效手段。

### 核心公式
```
你是一位{role}，拥有{years}年{years}经验，专业领域包括{expertise}。
你的工作方式是{work_style}。
你必须遵守的规则：{rules}。
```

In [ ]:
# ============================================================
# 角色框架模板库
# ============================================================

class RoleLibrary:
    """
    角色模板库：管理预定义的角色框架。
    
    What: 提供多种预定义的专家角色模板。
    Why: 角色框架是控制输出质量和风格的杠杆；预定义角色确保一致性。
    When: 构建多场景 LLM 应用，需要统一管理角色定义。
    """
    
    ROLES = {
        "code_reviewer": {
            "role": "资深代码审查专家",
            "years": 12,
            "expertise": ["代码质量", "安全漏洞", "性能优化", "设计模式"],
            "work_style": "逐行审查，先分析再给建议，使用具体例子说明",
            "rules": [
                "必须指出安全风险（XSS、SQL注入、敏感信息泄露等）",
                "给出的修改建议必须是可直接执行的代码",
                "区分必须修复（ERROR）和建议优化（WARN）",
            ],
            "output_format": """
【总体评价】评分：X/10
【ERROR】必须修复的问题
  1. [文件:行号] 问题描述 → 修复建议
【WARN】建议优化项
  1. [文件:行号] 优化建议
【安全性评估】
【性能评估】""",
        },
        "data_analyst": {
            "role": "资深数据分析师",
            "years": 8,
            "expertise": ["统计建模", "数据可视化", "A/B测试", "用户行为分析"],
            "work_style": "数据驱动，先假设再验证，所有结论必须有数据支撑",
            "rules": [
                "所有数字必须保留两位小数",
                "给出置信度或误差范围",
                "区分相关性和因果性",
            ],
            "output_format": """
【核心发现】1-2句话总结
【数据分析】
  - 描述性统计
  - 趋势分析
  - 对比分析
【结论与建议】
【局限性】本分析的限制和假设""",
        },
        "technical_writer": {
            "role": "技术文档撰写专家",
            "years": 10,
            "expertise": ["API文档", "用户手册", "技术博客", "架构文档"],
            "work_style": "先构建大纲，再逐步填充，确保逻辑流畅",
            "rules": [
                "使用主动语态，避免被动语态",
                "每个技术术语首次出现必须用中文解释",
                "代码示例必须完整可运行",
            ],
            "output_format": """
## 概述
## 前置条件
## 详细步骤
## 示例代码
## 常见问题""",
        },
    }
    
    # Jinja2 模板
    _TPL = jinja_env.from_string("""
你是一位{{ role }}，拥有{{ years }}年工作经验。

专业领域：{{ expertise | join('、') }}

工作方式：{{ work_style }}

必须遵守的规则：
{% for rule in rules %}
{{ loop.index }}. {{ rule }}
{% endfor %}

输出格式：
{{ output_format }}
""")
    
    @classmethod
    def build_prompt(
        cls,
        role_name: str,
        user_input: str,
        overrides: dict | None = None,
    ) -> str:
        """
        根据角色名构建 System Prompt。
        
        Args:
            role_name: 预定义角色名
            user_input: 用户输入
            overrides: 可覆盖的角色参数
        Returns:
            完整的 System Prompt
        """
        if role_name not in cls.ROLES:
            raise ValueError(f"未找到角色 '{role_name}'。可用角色: {list(cls.ROLES.keys())}")
        
        role_config = dict(cls.ROLES[role_name])  # 浅拷贝
        if overrides:
            role_config.update(overrides)
        
        return cls._TPL.render(**role_config)

# 测试角色库
print("=== 角色: code_reviewer ===")
print(RoleLibrary.build_prompt("code_reviewer", ""))

print("\n=== 角色: data_analyst (带覆盖) ===")
print(RoleLibrary.build_prompt("data_analyst", "", overrides={"years": 15}))

---
## 4. 动态 Few-shot 注入

### 原理
根据用户查询动态检索最相关的示例注入到提示中，而非使用固定的示例集。

### 注入策略
1. **静态注入**：始终使用同一组示例（最简单，但不灵活）
2. **语义检索注入**：用查询向量检索最相似的 K 个示例
3. **类别感知注入**：每个输出类别至少注入一个代表示例

In [ ]:
# ============================================================
# 动态 Few-shot 注入实现
# ============================================================

class DynamicFewShotInjector:
    """
    动态Few-shot注入器：根据查询选择最相关的示例。
    
    What: 维护示例库，通过语义相似度检索最相关的K个示例注入提示。
    Why: 固定示例往往无法覆盖所有场景，动态选择可以提升每个查询的匹配度。
    When: 示例库较大（>10个），且查询多样性高时使用。
    """
    
    def __init__(self, examples: list[dict]):
        """
        Args:
            examples: 示例列表，每个示例 {"input": str, "output": str, "category": str}
        """
        self.examples = examples
        self._template = jinja_env.from_string("""
以下是一些参考示例：

{% for ex in selected_examples %}
示例 {{ loop.index }}:
输入: {{ ex.input }}
输出: {{ ex.output }}

{% endfor %}
现在请处理以下输入：
输入: {{ query }}
输出:
""")
    
    def inject(
        self,
        query: str,
        k: int = 3,
        strategy: str = "similarity",
    ) -> str:
        """
        选择示例并注入到提示模板中。
        
        Args:
            query: 用户查询
            k: 选择的示例数量
            strategy: 选择策略
                - "similarity": 基于字符重叠的简单相似度
                - "category_balance": 每个类别均匀采样
                - "random": 随机选择
        Returns:
            注入示例后的 User Prompt 片段
        """
        if strategy == "similarity":
            selected = self._select_by_similarity(query, k)
        elif strategy == "category_balance":
            selected = self._select_by_category_balance(k)
        elif strategy == "random":
            import random
            selected = random.sample(self.examples, min(k, len(self.examples)))
        else:
            selected = self.examples[:k]
        
        return self._template.render(selected_examples=selected, query=query)
    
    def _select_by_similarity(self, query: str, k: int) -> list[dict]:
        """
        基于 Jaccard 字符相似度选择示例。
        
        Why: 纯字符级别，无需外部嵌入模型，适合快速原型。
        """
        query_chars = set(query)
        scored = []
        for ex in self.examples:
            ex_chars = set(ex["input"])
            # Jaccard 相似度
            intersection = len(query_chars & ex_chars)
            union = len(query_chars | ex_chars)
            score = intersection / union if union > 0 else 0
            scored.append((score, ex))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [ex for _, ex in scored[:k]]
    
    def _select_by_category_balance(self, k: int) -> list[dict]:
        """
        按类别平衡选择，确保每个类别都被覆盖。
        
        Why: 避免多分类任务中某些类别缺少示例。
        """
        from collections import defaultdict
        by_cat = defaultdict(list)
        for ex in self.examples:
            by_cat[ex.get("category", "default")].append(ex)
        
        selected = []
        cats = list(by_cat.keys())
        for i in range(k):
            cat = cats[i % len(cats)]
            if by_cat[cat]:
                selected.append(by_cat[cat].pop(0))
        return selected

# 构建示例库
support_examples = [
    {"input": "我想订一张去北京的机票", "output": '{"intent": "booking", "entity": "flight"}', "category": "booking"},
    {"input": "帮我查一下天气", "output": '{"intent": "inquiry", "entity": "weather"}', "category": "inquiry"},
    {"input": "这个产品有问题，我要退货", "output": '{"intent": "complaint", "entity": "refund"}', "category": "complaint"},
    {"input": "之前的订单能改一下吗", "output": '{"intent": "modification", "entity": "order"}', "category": "modification"},
    {"input": "帮我预约明天下午的会议室", "output": '{"intent": "booking", "entity": "meeting_room"}', "category": "booking"},
    {"input": "我想问一下你们的工作时间", "output": '{"intent": "inquiry", "entity": "business_hours"}', "category": "inquiry"},
    {"input": "我要投诉你们的客服态度很差", "output": '{"intent": "complaint", "entity": "service"}', "category": "complaint"},
    {"input": "把明天3点的会议改到4点", "output": '{"intent": "modification", "entity": "meeting"}', "category": "modification"},
]

injector = DynamicFewShotInjector(support_examples)

print("=== 策略1: 相似度选择 ===")
result_sim = injector.inject("帮我订一张明天去上海的机票", k=3, strategy="similarity")
print(result_sim)

print("=== 策略2: 类别平衡选择 ===")
result_bal = injector.inject("我要投诉服务", k=3, strategy="category_balance")
print(result_bal)

print("=== 策略3: 随机选择 ===")
result_rand = injector.inject("查询订单状态", k=3, strategy="random")
print(result_rand)

---
## 5. 模板继承与组合

### 模式
通过"基础模板 + 扩展块"实现模板复用，类似面向对象中的基类和派生类。

In [ ]:
# ============================================================
# 模板继承和组合
# ============================================================

# 基础模板
base_template_str = """
你是一个{{ role }}。

{% block constraints %}
请简洁准确地回答问题。
{% endblock %}

{% block output_format %}
请以纯文本格式回答。
{% endblock %}

{% block extra %}
{% endblock %}
"""

class ComposablePrompt:
    """
    可组合提示构建器：支持模板继承和块覆盖。
    
    What: 基于 Jinja2 块的模板组合系统。
    Why: 避免重复编写相似提示，通过继承和组合实现代码复用。
    When: 多个场景共享90%的提示结构，只有10%不同。
    """
    
    def __init__(self, base_template: str):
        self._base = base_template
        self._overrides: dict[str, str] = {}  # block_name → content
    
    def override_block(self, block_name: str, content: str) -> "ComposablePrompt":
        """覆盖模板中的某个块。"""
        self._overrides[block_name] = content
        return self  # 链式调用
    
    def compose(self, role: str, extra_context: dict | None = None) -> str:
        """
        组合所有块，生成最终提示。
        
        Args:
            role: 角色名称
            extra_context: 额外上下文
        Returns:
            组合后的提示文本
        """
        # 简单实现：用字符串替换模拟块继承
        prompt = self._base.replace(
            "{% block constraints %}\n请简洁准确地回答问题。\n{% endblock %}",
            self._overrides.get("constraints", "\n请简洁准确地回答问题。\n"),
        )
        prompt = prompt.replace(
            "{% block output_format %}\n请以纯文本格式回答。\n{% endblock %}",
            self._overrides.get("output_format", "\n请以纯文本格式回答。\n"),
        )
        prompt = prompt.replace(
            "{% block extra %}\n{% endblock %}",
            self._overrides.get("extra", ""),
        )
        
        # 使用 Jinja2 渲染剩余变量
        tpl = jinja_env.from_string(prompt)
        return tpl.render(role=role, **(extra_context or {}))

# 演示：三个相关但不完全相同的提示

# 场景1：代码生成助手
code_engineer = ComposablePrompt(base_template_str).override_block(
    "constraints",
    """
代码规范要求：
1. 使用 Python 3.10+ 语法（类型注解、match-case）
2. 包含完整的 docstring 和类型注解
3. 异常处理必须完善
4. 复杂度尽可能低
"""
).override_block(
    "output_format",
    """
输出格式：
【思路】解释解题思路
【代码】完整可运行代码
【复杂度】时间和空间复杂度
【测试】使用示例
"""
)

print("=== 代码生成助手 ===")
print(code_engineer.compose("Python 开发专家"))

# 场景2：文档写作助手（共享基础结构，覆盖 output_format）
doc_writer = ComposablePrompt(base_template_str).override_block(
    "constraints",
    """
写作要求：
1. 语言简洁明了，避免啰嗦
2. 使用具体例子说明抽象概念
3. 适当使用图表描述（用 ASCII 艺术代替）
"""
).override_block(
    "output_format",
    """
输出格式（Markdown）：
## 概述
## 核心概念
## 使用示例
## 注意事项
"""
).override_block(
    "extra",
    """
注意：如果涉及代码，请使用代码块标注语言类型。
"""
)

print("\n=== 文档写作助手 ===")
print(doc_writer.compose("技术文档撰写专家"))

# 场景3：仅继承基础模板的简单助手
simple_assistant = ComposablePrompt(base_template_str)
print("\n=== 基础助手（无覆盖）===")
print(simple_assistant.compose("AI 助手"))

---
## 6. 常见反模式

### 反模式 1: 硬编码提示字符串
```python
# 差 ❌
prompt = "你是一个客服，请回答:" + user_input + "请用JSON格式返回"

# 好 ✓
prompt = template.render(role="客服", user_input=user_input)
```

### 反模式 2: 字符串拼接
```python
# 差 ❌ — 难以阅读和维护
system = "你是" + role + "\n你的工作是" + task + "\n规则：" + "\n".join(rules)

# 好 ✓ — 使用 Jinja2 模板
system = template.render(role=role, task=task, rules=rules)
```

### 反模式 3: 缺少输入净化
```python
# 差 ❌ — 用户输入可能包含"{{ }}"导致 Jinja2 错误或注入
return tpl.render(user_input=user_raw_input)

# 好 ✓ — 自动转义或使用 safe 过滤器前的检查
return tpl.render(user_input=user_raw_input)  # autoescape=True 自动处理
```

### 反模式 4: 过长的 System Prompt
System Prompt 超过 500 tokens 后，模型对指令的遵循度下降。应将业务规则移到 User Prompt 中，或用 Few-shot 示例代替冗长的规则描述。

In [ ]:
# ============================================================
# 反模式演示与修复
# ============================================================

# 反模式 1：硬编码修复

# 差的做法
def bad_prompt_builder(user_input: str, language: str) -> str:
    return (
        "你是一个翻译助手。将以下文本翻译为" + language + "。\n"
        "文本：" + user_input + "\n"
        "翻译："
    )

# 好的做法
TRANSLATION_TPL = jinja_env.from_string("""
你是一个翻译助手。
将以下文本翻译为{{ target_language }}。

文本：{{ text }}
翻译：
""")

def good_prompt_builder(user_input: str, language: str) -> str:
    return TRANSLATION_TPL.render(target_language=language, text=user_input)

print("=== 反模式对比 ===")
sample_input = "Artificial Intelligence is transforming every industry."
print("差: ", bad_prompt_builder(sample_input, "中文"))
print("\n好: ", good_prompt_builder(sample_input, "中文"))

# 反模式 4 的量化：检测 System Prompt 长度
def estimate_tokens_rough(text: str) -> int:
    """粗略估算 token 数（中文字符约 1.5 token，英文单词约 1.3 token）"""
    chinese_chars = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
    english_words = len(text.split()) - chinese_chars // 2
    return int(chinese_chars * 1.5 + english_words * 1.3)

# 测试一个过长的 System Prompt
overly_long_system = "你是一个专业的AI助手。" * 200  # 刻意制造长提示
tokens = estimate_tokens_rough(overly_long_system)
print(f"\n=== Token 长度测试 ===")
print(f"字符数: {len(overly_long_system)}")
print(f"估算 Token: {tokens}")
print(f"状态: {'⚠️ 过长! (>500 tokens)' if tokens > 500 else '✓ 合理'}")

In [ ]:
# ============================================================
# 综合练习：构建完整的提示模板系统
# ============================================================

class PromptTemplateSystem:
    """
    完整的提示模板系统：整合角色库、模板引擎、动态 Few-shot。
    
    What: 将本 notebook 的所有概念整合为一个统一的提示构建系统。
    Why: 展示各组件如何协作，形成生产级别的提示构建管道。
    When: 构建实际 LLM 应用时的起点。
    """
    
    def __init__(self):
        self.role_library = RoleLibrary()
        self.injector: DynamicFewShotInjector | None = None
        
        # 最终组合模板
        self._master_tpl = jinja_env.from_string("""
{{ system_prompt }}
""")
    
    def configure_examples(self, examples: list[dict]):
        """配置 Few-shot 示例库"""
        self.injector = DynamicFewShotInjector(examples)
    
    def build(
        self,
        role_name: str,
        user_input: str,
        use_fewshot: bool = True,
        k_shots: int = 3,
        role_overrides: dict | None = None,
    ) -> tuple[str, str]:
        """
        构建完整的 System + User 提示。
        
        Returns:
            (system_prompt, user_prompt)
        """
        # 1. 构建 System Prompt
        system_prompt = RoleLibrary.build_prompt(
            role_name, user_input, overrides=role_overrides
        )
        
        # 2. 构建 User Prompt（可选 Few-shot）
        if use_fewshot and self.injector and self.injector.examples:
            user_prompt = self.injector.inject(user_input, k=k_shots)
        else:
            user_prompt = f"输入: {user_input}\n输出:"
        
        return system_prompt, user_prompt

# 实例化并测试
pts = PromptTemplateSystem()
pts.configure_examples(support_examples)

sys_p, usr_p = pts.build(
    role_name="code_reviewer",
    user_input="帮我审查这段代码的安全性: SELECT * FROM users WHERE name = '" + username + "'",
    use_fewshot=True,
    k_shots=2,
)

print("=== 最终 System Prompt ===")
print(sys_p)
print("\n=== 最终 User Prompt ===")
print(usr_p)

print("\n提示模板系统 Build 成功！")

## 本节小结

1. **Jinja2 模板** 是提示工程的基础设施，提供变量、条件和循环
2. **System / User 分离** 让模型"是什么"和"做什么"各司其职
3. **角色框架** 是最有效的单点提升技巧
4. **动态 Few-shot** 比静态 Few-shot 更精准，尤其适合多样的查询
5. **模板继承** 减少重复，让相似但不同的提示共享基础结构
6. **避免反模式**：不硬编码、不拼接字符串、不过长 System Prompt